# Phase 2c (v2) — SFT-LoRA Training, Fixed for 40GB + Completion-Only Loss

Same fixes as Phase 2b v2 (completion-only loss, left-truncation, 40GB-friendly config), but **target = gold_summary** instead of teacher_summary. This is the SFT baseline that the KD-trained student will be compared against.

**Important:** everything except `target_field` and `output_dir` is identical to Phase 2b v2 — same data, same hyperparameters, same LoRA config, same seed. Only the supervision signal differs.

In [1]:
!pip install -q transformers==4.46.0 datasets==2.21.0 accelerate==1.0.1 peft==0.13.2 trl==0.11.4 sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.8 MB/s eta 0:00:00
Reason for being yanked: This version unfortunately does not work with 3.8 but we did not drop the support yet
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.9/330.9 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.6/316.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 111.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 17.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages 

In [3]:
import torch, json
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig, DataCollatorForCompletionOnlyLM
from transformers import EarlyStoppingCallback

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

CONFIG = {
    'student_model': 'Qwen/Qwen2.5-0.5B',
    'teacher_data': 'teacher_generations.jsonl',
    'output_dir': './model_outputs/student_sft_lora',   # new dir so 10k results aren't overwritten
    'target_field': 'gold_summary',
    'num_epochs': 3,                                          # was 4 → less needed at 50k
    'per_device_batch_size': 4,
    'gradient_accumulation_steps': 4,                         # effective batch 16
    'learning_rate': 2e-4,
    'warmup_ratio': 0.03,
    'max_seq_length': 1536,
    'max_article_chars': 6000,
    'lora_r': 32,
    'lora_alpha': 64,
    'lora_dropout': 0.1,
    'weight_decay': 0.01,
    'eval_split_size': 500,                                   # was 200 → more reliable val signal at scale
    'seed': 42,
}
Path(CONFIG['output_dir']).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, indent=2))

GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB
{
  "student_model": "Qwen/Qwen2.5-0.5B",
  "teacher_data": "teacher_generations.jsonl",
  "output_dir": "./model_outputs/student_sft_lora",
  "target_field": "gold_summary",
  "num_epochs": 3,
  "per_device_batch_size": 4,
  "gradient_accumulation_steps": 4,
  "learning_rate": 0.0002,
  "warmup_ratio": 0.03,
  "max_seq_length": 1536,
  "max_article_chars": 6000,
  "lora_r": 32,
  "lora_alpha": 64,
  "lora_dropout": 0.1,
  "weight_decay": 0.01,
  "eval_split_size": 500,
  "seed": 42
}


In [4]:
records = []
with open(CONFIG['teacher_data']) as f:
    for line in f:
        records.append(json.loads(line))
print(f'Loaded {len(records)} records')
print('Sample target ({}): {}'.format(CONFIG['target_field'], records[0][CONFIG['target_field']]))

Loaded 50000 records
Sample target (gold_summary): John and .
Audrey Cook were discovered alongside their daughter, Maureen .
They were found at Tremarle Home Park in Cornwall .
Investigators say the three died of carbon monoxide .
poisoning .


In [5]:
tokenizer = AutoTokenizer.from_pretrained(CONFIG['student_model'])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'
tokenizer.truncation_side = 'left'

SYSTEM_PROMPT = (
    'You are a concise news summarizer. Write a short summary of the article in 2-3 sentences. '
    'Output only the summary itself, with no preamble, headers, or commentary.'
)
USER_TEMPLATE = 'Article:\n{article}\n\nSummary:'

def format_example(rec) -> dict:
    article = rec['article']
    if len(article) > CONFIG['max_article_chars']:
        article = article[-CONFIG['max_article_chars']:]
    msgs = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': USER_TEMPLATE.format(article=article)},
        {'role': 'assistant', 'content': rec[CONFIG['target_field']]},
    ]
    return {'text': tokenizer.apply_chat_template(msgs, tokenize=False)}

formatted = [format_example(r) for r in records]
full_ds = Dataset.from_list(formatted)
split = full_ds.train_test_split(test_size=CONFIG['eval_split_size'], seed=CONFIG['seed'])
train_ds, eval_ds = split['train'], split['test']
print(f'Train: {len(train_ds)}  Eval: {len(eval_ds)}')
print('\n--- Sample formatted text (end of sequence) ---')
print(formatted[0]['text'][-500:])

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Train: 49500  Eval: 500

--- Sample formatted text (end of sequence) ---
 and walk her dog, a little Jack Russell, it is so sad . what has happened, I understand the dog went with them. 'They . will be sorely missed and I think everyone is just in shock at the . moment, I would like to send my condolences to the Cook family.'

Summary:<|im_end|>
<|im_start|>assistant
John and .
Audrey Cook were discovered alongside their daughter, Maureen .
They were found at Tremarle Home Park in Cornwall .
Investigators say the three died of carbon monoxide .
poisoning .<|im_end|>



In [6]:
RESPONSE_TEMPLATE = '<|im_start|>assistant\n'
response_ids = tokenizer.encode(RESPONSE_TEMPLATE, add_special_tokens=False)
print(f'Response template token ids: {response_ids}')
assert RESPONSE_TEMPLATE in formatted[0]['text'], 'response template not found'
print('✓ Response template found in formatted examples')

collator = DataCollatorForCompletionOnlyLM(
    response_template=RESPONSE_TEMPLATE,
    tokenizer=tokenizer,
)

Response template token ids: [151644, 77091, 198]
✓ Response template found in formatted examples


In [7]:
model = AutoModelForCausalLM.from_pretrained(
    CONFIG['student_model'], torch_dtype=torch.bfloat16, device_map=DEVICE)
model.config.use_cache = False
model.gradient_checkpointing_enable()
if hasattr(model, 'enable_input_require_grads'):
    model.enable_input_require_grads()

lora_config = LoraConfig(
    r=CONFIG['lora_r'],
    lora_alpha=CONFIG['lora_alpha'],
    lora_dropout=CONFIG['lora_dropout'],
    bias='none',
    task_type=TaskType.CAUSAL_LM,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

trainable params: 17,596,416 || all params: 511,629,184 || trainable%: 3.4393


In [9]:
sft_config = SFTConfig(
    output_dir=CONFIG['output_dir'],
    num_train_epochs=CONFIG['num_epochs'],
    per_device_train_batch_size=CONFIG['per_device_batch_size'],
    per_device_eval_batch_size=CONFIG['per_device_batch_size'],
    gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],
    gradient_checkpointing=False,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    learning_rate=CONFIG['learning_rate'],
    lr_scheduler_type='cosine',
    warmup_ratio=CONFIG['warmup_ratio'],
    bf16=True,
    optim='adamw_torch_fused',
    logging_steps=20,

    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    eval_strategy='steps',
    eval_steps=200,
    save_strategy='steps',
    save_steps=200,
    weight_decay=0.01,
    save_total_limit=2,
    max_seq_length=CONFIG['max_seq_length'],
    dataset_text_field='text',
    packing=False,
    report_to='none',
    seed=CONFIG['seed'],
)

trainer = SFTTrainer(
    model=model, args=sft_config,
    train_dataset=train_ds, eval_dataset=eval_ds,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    tokenizer=tokenizer, data_collator=collator,
)

# Sanity check
print('Running 1-step sanity check...')
batch = next(iter(trainer.get_train_dataloader()))
batch = {k: v.to(DEVICE) for k, v in batch.items()}
out = model(**batch)
print(f'  Initial loss: {out.loss.item():.4f}')
n_unmasked = (batch['labels'] != -100).sum().item()
n_total = batch['labels'].numel()
print(f'  Unmasked label tokens: {n_unmasked} / {n_total} ({100*n_unmasked/n_total:.1f}%)')
del out, batch; torch.cuda.empty_cache()

Map:   0%|          | 0/49500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:401: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  super().__init__(


Running 1-step sanity check...
  Initial loss: 2.6083
  Unmasked label tokens: 360 / 4948 (7.3%)


In [13]:

import torch
import functools

# Save original
_original_torch_load = torch.load

# Patch to default weights_only=False
@functools.wraps(_original_torch_load)
def _patched_torch_load(*args, **kwargs):
    if 'weights_only' not in kwargs:
        kwargs['weights_only'] = False
    return _original_torch_load(*args, **kwargs)

torch.load = _patched_torch_load

# Now resume
trainer.train(resume_from_checkpoint='./model_outputs/student_sft_lora/checkpoint-3600')

trainer.save_model(CONFIG['output_dir'])
log_path = Path(CONFIG['output_dir']) / 'training_log.json'
with open(log_path, 'w') as f:
    json.dump(trainer.state.log_history, f, indent=2)
print(f'\nSaved adapter + log to {CONFIG["output_dir"]}')

Step,Training Loss,Validation Loss
3800,1.574700,1.789970



Saved adapter + log to ./model_outputs/student_sft_lora


In [14]:
from google.colab import userdata
from huggingface_hub import login, HfApi

# Reads from Colab Secrets (left sidebar → 🔑 icon → add secret named "HF_TOKEN")
login(token=userdata.get('HF_TOKEN'))

# Fill in your HF username
HF_USERNAME = 'Harsha901'
REPO_NAME = 'qwen2.5-0.5b-sft-lora-cnndm-50k'
REPO_ID = f'{HF_USERNAME}/{REPO_NAME}'

# Push the LoRA adapter and tokenizer
trainer.model.push_to_hub(
    REPO_ID,
    commit_message='SFT-LoRA student on CNN/DailyMail, best val_loss checkpoint',
    private=True,
)
tokenizer.push_to_hub(REPO_ID, private=True)

# Add a README with reproducibility info
readme = f"""---
base_model: Qwen/Qwen2.5-0.5B
library_name: peft
tags:
- lora
- summarization
- sft
- cnn_dailymail
---

# {REPO_NAME}

LoRA adapter for `Qwen/Qwen2.5-0.5B` fine-tuned via **supervised fine-tuning on gold human-written summaries** from the CNN/DailyMail summarization dataset.

This adapter is the SFT baseline in a KD-vs-SFT comparison study. The matched KD-trained counterpart trained on `Qwen/Qwen2.5-7B-Instruct` teacher generations is at `{HF_USERNAME}/qwen2.5-0.5b-kd-lora-cnndm`.

## Training config
- LoRA: r={CONFIG['lora_r']}, alpha={CONFIG['lora_alpha']}, dropout={CONFIG['lora_dropout']}
- Target modules: q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj
- Effective batch size: {CONFIG['per_device_batch_size'] * CONFIG['gradient_accumulation_steps']}
- Learning rate: {CONFIG['learning_rate']}, cosine schedule, {CONFIG['warmup_ratio']:.0%} warmup
- Weight decay: {CONFIG['weight_decay']}
- Max sequence length: {CONFIG['max_seq_length']}
- Trained for up to {CONFIG['num_epochs']} epochs with early stopping on `eval_loss`
- Best checkpoint loaded at end of training

## Usage
```python
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

base = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-0.5B', torch_dtype=torch.bfloat16)
model = PeftModel.from_pretrained(base, '{REPO_ID}')
tokenizer = AutoTokenizer.from_pretrained('{REPO_ID}')
```
"""
api = HfApi()
api.upload_file(
    path_or_fileobj=readme.encode(),
    path_in_repo='README.md',
    repo_id=REPO_ID,
    commit_message='Add README',
)

print(f'\n✓ Pushed to: https://huggingface.co/{REPO_ID}')

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  57%|#####6    | 40.0MB / 70.4MB            

No files have been modified since last commit. Skipping to prevent empty commit.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp1fmbnfwn/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.



✓ Pushed to: https://huggingface.co/Harsha901/qwen2.5-0.5b-sft-lora-cnndm-50k
